# 07 — Contextual Retrieval: does a written context line help a chunk?

**Concept:** before a chunk is embedded, prepend a short line that says where it comes from.
[Anthropic's Contextual Retrieval post](https://www.anthropic.com/engineering/contextual-retrieval)
has an LLM write that line from the whole document. The free control is to prepend the title.

## The prediction, written before running (2026-10-29)

> **The LLM context line beats plain chunks, but not the title.** Arm 3's paired nDCG@10 CI
> against arm 1 lies above 0. Its CI against arm 2 includes 0.
>
> NFCorpus documents are short abstracts (about 1.7k chars). What a chunk loses when it is cut
> out is mostly the title, and the title is free.

The prediction fails if arm 3 against arm 1 includes 0, or if arm 3 against arm 2 excludes 0 in
either direction.

## Setup, fixed before running

Every arm uses notebook 05's chunk-pool document vector and changes only the text of each chunk.
Same encoder (`all-MiniLM-L6-v2`), same chunks (400-char windows of the body, 80 overlap, 18,835
chunks over 3,633 docs), same blend (0.25 × title vector + 0.75 × mean of chunk vectors).

| arm | text embedded for each chunk |
|---|---|
| 1. plain | the chunk (notebook 05's `chunk_pool`; recomputed here and checked against the committed scores) |
| 2. title | the doc title, a newline, the chunk (docs with an empty title get no prefix) |
| 3. LLM context | a Claude Haiku context line, a newline, the chunk |

Arm 3's lines come from headless Claude Code (`claude -p --model haiku`), one call per document.
The call sees the title, the abstract and the numbered chunks. It returns one line per chunk,
following the post's prompt ("a short succinct context to situate this chunk within the overall
document for the purposes of improving search retrieval"). Every line is committed under
`results/context/`, so the notebook reruns without an LLM. A document whose reply cannot be parsed
after two retries keeps its plain chunks, and the count of those is reported.

Contextual BM25 is left out: BM25 lost to dense in notebook 03.

Scored on all 323 queries against the human qrels. `paired_bootstrap`, seed 42, 10,000 resamples,
corpus order. Reported alongside but not bet on: arm 2 against arm 1, and every arm against the
title+text dense baseline and against title-only.


In [1]:
import numpy as np

from ragexp.contextualize import load_contexts
from ragexp.data import load_nfcorpus
from ragexp.embed import Embedder
from ragexp.metrics import paired_bootstrap
from ragexp.retrieve import _chunks, pool_doc_matrix
from ragexp.runs import evaluate_run, load_scores, metric_vector, save_scores, summarize

corpus = load_nfcorpus()
qids = [q.query_id for q in corpus.queries]
doc_ids = corpus.doc_ids
emb = Embedder()

committed = {name: load_scores(name) for name in ("dense", "title_only", "chunk_pool")}
contexts = load_contexts("haiku_contexts")  # committed: no LLM call in this notebook
failed = [d for d in doc_ids if contexts.get(d) is None]
print(corpus.summary())
print(f"context lines: {len(contexts)} docs stored, {len(failed)} failed (those keep plain chunks)")
print(f"written by: {sorted({v['model'] for v in contexts.values() if v})}")

NFCorpus: 3633 docs, 323 queries, 12334 judgments (38.2/query), grade distribution {1: 11758, 2: 576}
context lines: 3633 docs stored, 0 failed (those keep plain chunks)
written by: ['claude-haiku-5-5']


## The three arms

Each arm hands `pool_doc_matrix` the same chunks and changes only what is prepended to each one.
Arm 1 prepends nothing. Arm 2 prepends the title. Arm 3 prepends the Haiku line written for that
chunk. The title vector in the 0.25 blend is the same in all three.

In [2]:
by_id = {d.doc_id: d for d in corpus.docs}
n_chunks = {d.doc_id: len(_chunks(d.text)) for d in corpus.docs}

title_prefixes = {d.doc_id: [d.title.strip()] * n_chunks[d.doc_id] for d in corpus.docs}
context_prefixes = {d: v["contexts"] for d, v in contexts.items() if v is not None}

matrices = {
    "arm1-plain": pool_doc_matrix(corpus.docs, emb, mode="pool"),
    "arm2-title": pool_doc_matrix(corpus.docs, emb, mode="pool", chunk_prefixes=title_prefixes),
    "arm3-context": pool_doc_matrix(corpus.docs, emb, mode="pool", chunk_prefixes=context_prefixes),
}

lines = [c for v in context_prefixes.values() for c in v]
print(f"chunks: {sum(n_chunks.values())}; context lines: {len(lines)}, "
      f"median {int(np.median([len(c) for c in lines]))} chars")
print(f"docs with an empty title (arm 2 adds nothing): {sum(1 for d in corpus.docs if not d.title.strip())}")

chunks: 18835; context lines: 18835, median 215 chars
docs with an empty title (arm 2 adds nothing): 0


### Does the prefix push the chunk past the encoder's limit?

`all-MiniLM-L6-v2` truncates its input. A prefix that is too long would cut off the end of the
chunk it is meant to help. This counts how often that happens in each arm.

In [3]:
tokenizer = emb._load().tokenizer
max_len = emb._load().max_seq_length


def clipped_share(prefixes):
    texts = [f"{p}\n{c}" if p else c
             for d in corpus.docs
             for p, c in zip(prefixes.get(d.doc_id, [""] * n_chunks[d.doc_id]), _chunks(d.text))]
    return float(np.mean([len(tokenizer.encode(t)) > max_len for t in texts]))


print(f"max_seq_length: {max_len}")
for name, prefixes in (("arm1-plain", {}), ("arm2-title", title_prefixes), ("arm3-context", context_prefixes)):
    print(f"  {name:<13} chunks clipped: {clipped_share(prefixes):.1%}")

max_seq_length: 256


  arm1-plain    chunks clipped: 0.0%


  arm2-title    chunks clipped: 0.0%


  arm3-context  chunks clipped: 0.0%


## Self-check: arm 1 is notebook 05's chunk pool

Arm 1 is the same code path as notebook 05's `chunk-pool`, so its per-query scores must match the
committed `results/scores/chunk_pool.json`. If they do not, the other two arms are not being
compared with what notebook 05 measured.

In [4]:
query_vecs = emb.encode([q.text for q in corpus.queries])


def rank_all(doc_matrix):
    sims = query_vecs @ doc_matrix.T
    return {qid: [doc_ids[j] for j in np.argsort(-sims[i])] for i, qid in enumerate(qids)}


run_scores = {name: evaluate_run(corpus, rank_all(m), recall_ks=(10, 50), ndcg_ks=(10,))
              for name, m in matrices.items()}

worst = max(abs(run_scores["arm1-plain"][q][m] - committed["chunk_pool"][q][m])
            for q in qids for m in ("recall@10", "recall@50", "ndcg@10")
            if committed["chunk_pool"][q][m] is not None)
assert worst < 1e-9, f"arm 1 drifted from nb 05's chunk_pool: {worst}"
print(f"self-check: arm 1 matches the committed chunk_pool scores (max |delta| = {worst})")

save_scores("chunk_title", run_scores["arm2-title"])
save_scores("chunk_context", run_scores["arm3-context"])

header = f"{'system':<14}" + "".join(f"{m:>12}" for m in ("recall@10", "recall@50", "ndcg@10"))
print("\n" + header)
print("-" * len(header))
rows = [("title-only", committed["title_only"]), ("dense", committed["dense"])] + list(run_scores.items())
for name, sc in rows:
    means = summarize(sc)
    print(f"{name:<14}" + "".join(f"{means[m]:>12.4f}" for m in ("recall@10", "recall@50", "ndcg@10")))

self-check: arm 1 matches the committed chunk_pool scores (max |delta| = 0.0)

system           recall@10   recall@50     ndcg@10
--------------------------------------------------
title-only          0.1355      0.2378      0.2885
dense               0.1550      0.2508      0.3167
arm1-plain          0.1536      0.2455      0.3140
arm2-title          0.1517      0.2507      0.3166
arm3-context        0.1566      0.2496      0.3191


## The two registered contrasts

Both are paired over the 323 queries in corpus order. The prediction needs the first CI above 0
and the second CI to include 0.

In [5]:
def compare(name_a, scores_a, name_b, scores_b):
    print(f"{name_b} vs {name_a}")
    for metric in ("recall@10", "recall@50", "ndcg@10"):
        a = metric_vector(scores_a, metric, qids)
        b = metric_vector(scores_b, metric, qids)
        delta, (lo, hi), p = paired_bootstrap(a, b)
        sig = "  <-- significant" if (lo > 0 or hi < 0) else ""
        print(f"  {metric:>10}: delta {delta:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}{sig}")
    print()


compare("arm1-plain", run_scores["arm1-plain"], "arm3-context", run_scores["arm3-context"])
compare("arm2-title", run_scores["arm2-title"], "arm3-context", run_scores["arm3-context"])

arm3-context vs arm1-plain
   recall@10: delta +0.0030  CI [-0.0010, +0.0076]  p=0.1556
   recall@50: delta +0.0041  CI [-0.0008, +0.0098]  p=0.1026
     ndcg@10: delta +0.0051  CI [-0.0011, +0.0117]  p=0.1078

arm3-context vs arm2-title
   recall@10: delta +0.0049  CI [-0.0071, +0.0171]  p=0.4092
   recall@50: delta -0.0011  CI [-0.0088, +0.0067]  p=0.7798
     ndcg@10: delta +0.0025  CI [-0.0077, +0.0131]  p=0.6486



## Reported alongside, not bet on

In [6]:
compare("arm1-plain", run_scores["arm1-plain"], "arm2-title", run_scores["arm2-title"])
for name in ("arm2-title", "arm3-context"):
    compare("dense", committed["dense"], name, run_scores[name])
    compare("title-only", committed["title_only"], name, run_scores[name])

arm2-title vs arm1-plain
   recall@10: delta -0.0019  CI [-0.0139, +0.0098]  p=0.7418
   recall@50: delta +0.0052  CI [-0.0015, +0.0124]  p=0.1274
     ndcg@10: delta +0.0026  CI [-0.0070, +0.0120]  p=0.5962

arm2-title vs dense
   recall@10: delta -0.0033  CI [-0.0179, +0.0121]  p=0.6418


   recall@50: delta -0.0000  CI [-0.0108, +0.0115]  p=0.9636


     ndcg@10: delta -0.0001  CI [-0.0134, +0.0133]  p=0.9646

arm2-title vs title-only
   recall@10: delta +0.0162  CI [+0.0080, +0.0261]  p=0.0000  <-- significant
   recall@50: delta +0.0129  CI [+0.0037, +0.0231]  p=0.0060  <-- significant
     ndcg@10: delta +0.0281  CI [+0.0190, +0.0380]  p=0.0000  <-- significant

arm3-context vs dense
   recall@10: delta +0.0016  CI [-0.0075, +0.0127]  p=0.8114
   recall@50: delta -0.0012  CI [-0.0109, +0.0093]  p=0.7850
     ndcg@10: delta +0.0024  CI [-0.0093, +0.0145]  p=0.7208

arm3-context vs title-only
   recall@10: delta +0.0211  CI [+0.0063, +0.0372]  p=0.0044  <-- significant


   recall@50: delta +0.0118  CI [-0.0007, +0.0245]  p=0.0642
     ndcg@10: delta +0.0306  CI [+0.0160, +0.0462]  p=0.0000  <-- significant



## Resolving the prediction

The prediction was **"the LLM context line beats plain chunks, but not the title"**. The second
half held. The first half failed: the context lines did not beat plain chunks either.

| arm | nDCG@10 | Δ vs arm 1 (paired) | Δ vs arm 2 (paired) |
|---|---|---|---|
| title-only (nb 05) | 0.2885 | | |
| dense, title + text (nb 02) | 0.3167 | | |
| 1. plain chunks | 0.3140 | | |
| 2. title + chunk | 0.3166 | +0.0026, CI [−0.0070, +0.0120], p = 0.60 | |
| 3. Haiku context + chunk | 0.3191 | **+0.0051, CI [−0.0011, +0.0117], p = 0.11** | +0.0025, CI [−0.0077, +0.0131], p = 0.65 |

1. **The context lines bought nothing measurable.** Haiku wrote 18,835 of them, one call per
   document, with no failures. Arm 3 comes out highest of the three, but by +0.005 nDCG@10 with a
   CI that includes 0. On recall@10 and recall@50 it nulls as well.

2. **The free control nulled too.** Prepending the title to each chunk moved nDCG@10 by +0.003.
   One likely reason: the pooled vector already carries the title through its 0.25 title blend, so
   a title prefix on every chunk repeats what the document vector already holds.

3. **No chunk arm beats embedding the whole document once.** All three tie the title+text dense
   baseline (every CI includes 0). These abstracts are short: about five chunks each, and not one
   chunk, prefixed or not, was clipped by the encoder. That is the regime where the September
   pickup note expected the context to have nothing to restore. The post's gains came from long,
   multi-section documents, which NFCorpus does not have.

**Not tested here:** contextual BM25, reranking on top, a different encoder, per-chunk calls (this
notebook makes one call per document), and the 2024 Claude model the post used. The run took
1,935 s with 8 calls in parallel. Its token cost was not recorded.

## Why the context line did not help: what the sources say (post-hoc)

Written after the run, from a literature check on 2026-10-29. It explains the result. It does not
re-score anything.

1. **NFCorpus is the kind of benchmark where context is not expected to help.** Conti, Faysse,
   Viaud et al. ([2505.24782](https://arxiv.org/abs/2505.24782)) built the ConTEB benchmark because
   "Existing benchmarks often rely on (or assume) self-contained document chunks. This creates a
   misleading perception that contextualization offers little to no benefit". Their Table 1 marks
   NanoBEIR, a small sample of BEIR datasets that has an NFCorpus split, "No context is needed".
   Their Table 2 then measures exactly this split, with one embedder (`modernbert-embed-large`) and
   Qwen-2.5-7B writing the context: Anthropic-style contextualization lifts the ConTEB average from
   52.0 to 72.4 nDCG@10 and leaves NanoBEIR at 63.2, unchanged to the decimal. That is the pattern
   this notebook found.
2. **The post measured a different retrieval unit.** Anthropic's numbers are chunk-retrieval
   failure rates on codebases, fiction, arXiv and science papers: each chunk is retrieved on its
   own. Here no chunk is retrieved alone. Notebook 05's design averages the chunk vectors into one
   document vector and blends in the title vector. A context line restates document-level
   information that the other chunks and the title already put into that vector. The pooled design
   is close to the worst case for this mechanism.

What the result does say: on short abstracts with pooled document vectors, an LLM context line adds
nothing measurable. What it does not say: anything about chunk-level retrieval on long documents.

**Open test, not run:** the post's own unit, a chunk ranked on its own with the document scored by
its best chunk, plain against context lines. The committed lines in `results/context/` make it
possible without new LLM calls. It needs its own written prediction first.

## Chunk-level arm. The prediction, written before running (2026-10-29)

The cell above names the gap between this notebook and the Anthropic post: the post ranks chunks on
their own, and this notebook pooled them into one document vector. This arm runs the post's unit.

> **In chunk-level retrieval the context line beats plain chunks.** Arm B's paired nDCG@10 CI
> against arm A lies above 0.

The prediction fails if the CI includes 0 or lies below it.

Why a win here when ConTEB found none on NanoBEIR: their Table 1 lists NanoBEIR at 1 chunk per
document, so no chunk is ever cut off from its document. NFCorpus documents here split into about
five chunks, and with each chunk ranked alone, chunks after the first carry no title.

**Setup, fixed before running.** The chunks, encoder (`all-MiniLM-L6-v2`) and Haiku lines from the
arms above, unchanged; no LLM call. Each chunk is embedded as its own vector. A query scores a
document as the maximum cosine over that document's chunks, and documents are ranked by that score.

| arm | text embedded for each chunk |
|---|---|
| A. plain, max | the chunk |
| B. context, max | the Haiku context line, a newline, the chunk |
| C. title, max (reported) | the doc title, a newline, the chunk |

Scored on all 323 queries against the human qrels. `paired_bootstrap`, seed 42, 10,000 resamples,
corpus order. Reported, not bet on: arm C against A and B, every max arm against `dense`
(title+text), and each max arm against its pooled twin above.